In [ ]:
# Data root is configurable: export SYNERGPCR_BASE=/path/to/released/tables
# (defaults to ./data). All input paths below are resolved against it.
from pathlib import Path
import os
import pandas as pd
import re
import time
import zipfile
import gzip
import json
from tqdm import tqdm
import concurrent.futures
import google.generativeai as genai
from typing_extensions import TypedDict

BASE = Path(os.environ.get("SYNERGPCR_BASE", "./data"))


In [ ]:
# ==========================================
# 1. API Configuration
# ==========================================
import os
API_KEY = os.environ.get("GEMINI_API_KEY")
genai.configure(api_key=API_KEY)

# Define the strict output schema for the LLM
class AssayExtraction(TypedDict):
    assay_type: str
    target_moa: str
    is_negation: bool
    disease_context: str
    cell_line: str

# Initialize the model with gemini-2.5-flash for higher reasoning and structured output
model = genai.GenerativeModel(
    'models/gemini-2.5-flash',
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": AssayExtraction,
        "temperature": 0.1 # Low temperature for factual extraction
    }
)

In [ ]:
# ==========================================
# 2. Helper Functions
# ==========================================
def flatten_and_join(value):
    """Flattens list elements and joins them into a single string."""
    if isinstance(value, list):
        flattened = []
        for item in value:
            if isinstance(item, list):
                flattened.extend(item)
            else:
                flattened.append(str(item))
        return ' '.join(flattened)
    elif isinstance(value, str):
        return value
    return str(value)

def generate_llm_prompt(title, description):
    """Constructs a prompt for structured assay information extraction."""
    prompt = f"""
    You are an expert bioinformatician and data curator.
    Analyze the following assay Title and Description to extract specific biological and pharmacological metadata.
    
    Extraction Rules:
    - assay_type: Choose from 'cAMP', 'Ca2+', 'IP1', 'GTPgS', 'beta-arrestin', 'binding_only', or 'reporter_other'.
    - target_moa: Choose from 'Agonist', 'Antagonist', 'Partial Agonist', 'Inverse Agonist', 'PAM', 'NAM', or 'Unknown'.
    - is_negation: Set to true ONLY if the assay explicitly states it failed to show the target MoA.
    - disease_context: The target disease, clinical indication, or therapeutic area. Return 'N/A' if not explicitly stated.
    - cell_line: The specific cell line used for the assay (e.g., HEK293, CHO, HeLa). Return 'N/A' if not explicitly stated.
    
    Assay Title: "{title}"
    Assay Description: "{description}"
    """
    return prompt

def extract_assay_info_llm(title, description, retries=3):
    """Calls the LLM API with error handling and retry logic."""
    prompt = generate_llm_prompt(title, description)
    
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            return json.loads(response.text)
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                time.sleep(5)
            else:
                print(f"Extraction error: {e}")
                break
                
    # Fallback dictionary matching the exact schema if all retries fail
    return {
        "assay_type": "Unknown", 
        "target_moa": "Unknown", 
        "is_negation": False,
        "disease_context": "N/A",
        "cell_line": "N/A"
    }

In [ ]:
# ==========================================
# 3. Main Processing Function
# ==========================================
def process_aid_llm(aid, aid_to_uniprot_gene_name, json_dir):
    """Processes each AID by extracting relevant information and calling the LLM."""
    start = ((aid - 1) // 1000) * 1000 + 1
    end = start + 999
    zip_file_name = f"{start:07d}_{end:07d}.zip"
    zip_file_path = os.path.join(json_dir, zip_file_name)
    json_file_name = f"{aid}.json.gz"
    
    if not os.path.exists(zip_file_path):
        return None
    
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zf:
            zip_file_list = zf.namelist()
            found_name = next((name for name in zip_file_list if name.endswith(f"/{json_file_name}") or name == json_file_name or name.endswith(f"\\{json_file_name}")), None)
            
            if found_name is None:
                return None
                
            with zf.open(found_name) as json_file:
                with gzip.open(json_file, 'rt', encoding='utf-8') as gz:
                    data = json.load(gz)
    except Exception as e:
        return None
    
    try:
        assay_descr = data['PC_AssaySubmit']['assay']['descr']
    except (KeyError, IndexError):
        return None
    
    assay_name = flatten_and_join(assay_descr.get('name', ''))
    assay_description = flatten_and_join(assay_descr.get('description', ''))
    assay_comments = flatten_and_join(assay_descr.get('comment', ''))
    
    gene_name, taxonomy, accession = "", "Not specified", "Not specified"
    if "target" in assay_descr and assay_descr['target']:
        target = assay_descr['target'][0]
        gene_name = target.get('name', '')
        accession = target.get('mol_id', {}).get('protein_accession', 'Not specified')
        taxonomy = target.get('organism', {}).get('org', {}).get('taxname', 'Not specified')
    
    target_receptor = aid_to_uniprot_gene_name.get(aid, '').lower()
    
    text_to_search = ' '.join([assay_name, assay_description, assay_comments])
    active_criteria = 'Not specified'
    match = re.search(r'(\d+\.?\d*)\s*(nM|μM|uM|mM)', text_to_search, re.IGNORECASE)
    if match:
        active_criteria = f"{match.group(1)}{match.group(2)}"

    # LLM Call
    combined_desc = f"{assay_description} {assay_comments}"
    llm_result = extract_assay_info_llm(assay_name, combined_desc)
    
    return {
        'AID': aid,
        'Gene_Name': gene_name,
        'Target_Receptor': target_receptor,
        'Accession': accession,
        'Taxonomy': taxonomy,
        'Assay_Type_LLM': llm_result.get('assay_type', 'Unknown'),
        'Target_MoA_LLM': llm_result.get('target_moa', 'Unknown'),
        'Is_Negation': llm_result.get('is_negation', False),
        'Disease_Context': llm_result.get('disease_context', 'N/A'),
        'Cell_Line': llm_result.get('cell_line', 'N/A'),
        'Active_Criteria': active_criteria
    }

In [ ]:
# ==========================================
# 4. Main Execution Block (Parallel)
# ==========================================
if __name__ == "__main__":
    df = pd.read_csv('./DB/PubChem/tb_aid_act_gpcr.csv', dtype={'GENE_ID': str})
    aid_to_uniprot_gene_name = df.dropna(subset=['UNIPROT_GENE_NAME']).set_index('AID')['UNIPROT_GENE_NAME'].str.lower().to_dict()
    aids = df['AID'].unique().tolist()
    
    json_dir = str(BASE / "DB/PubChem/pubchem_bioassay_json") 
    
    def process_single_aid(aid):
        return process_aid_llm(aid, aid_to_uniprot_gene_name, json_dir)

    print(f"Starting rigorous processing of {len(aids)} AIDs with Gemini 2.5 Flash...")
    aid_data = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_single_aid, aid): aid for aid in aids}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(aids), desc='Mining BioAssays'):
            result = future.result()
            if result:
                aid_data.append(result)

    output_df = pd.DataFrame(aid_data)
    
    output_dir = './Output/DB/PubChem/NAR'
    os.makedirs(output_dir, exist_ok=True)
    output_df.to_csv(os.path.join(output_dir, 'aid_information_llm_parsed_v2.csv'), index=False)
    print("\nProcessing complete. Saved to 'aid_information_llm_parsed_v2.csv'")

ID standardization

In [ ]:
import pandas as pd
import requests
import urllib.parse
import time
from tqdm import tqdm
import os

# ==========================================
# 1. API Mapping Functions
# ==========================================

def get_disease_ontology(disease_str):
    """
    EBI OLS4 API를 사용하여 질병명을 MONDO 또는 EFO 표준 ID로 변환합니다.
    """
    if pd.isna(disease_str) or str(disease_str).strip() == 'N/A' or not str(disease_str).strip():
        return None, None
        
    query = urllib.parse.quote(str(disease_str).strip())
    url = f"https://www.ebi.ac.uk/ols4/api/search?q={query}&ontology=mondo,efo&exact=false&rows=1"
    
    for attempt in range(3):
        try:
            res = requests.get(url, timeout=10)
            if res.status_code == 200:
                data = res.json()
                if data.get('response', {}).get('numFound', 0) > 0:
                    doc = data['response']['docs'][0]
                    # return: Standardized Name, Ontology ID (e.g., Alzheimer's disease, MONDO_0004975)
                    return doc.get('label'), doc.get('short_form') 
            elif res.status_code == 429:
                time.sleep(2)
            else:
                break
        except Exception as e:
            time.sleep(1)
            
    return None, None

def get_cell_line_ontology(cell_str):
    """
    ExPASy Cellosaurus API를 사용하여 세포주를 표준 CVCL ID로 변환합니다.
    """
    if pd.isna(cell_str) or str(cell_str).strip() == 'N/A' or not str(cell_str).strip():
        return None, None
        
    # API 혼동을 방지하기 위해 'cells', 'cell' 등 불필요한 단어 제거
    search_term = str(cell_str).lower().replace('cells', '').replace('cell', '').strip()
    query = urllib.parse.quote(search_term)
    url = f"https://api.cellosaurus.org/search/cell-line?q={query}&format=json"
    
    for attempt in range(3):
        try:
            res = requests.get(url, timeout=10)
            if res.status_code == 200:
                data = res.json()
                if 'search-results' in data and len(data['search-results']) > 0:
                    best_match = data['search-results'][0]
                    # return: Standardized Name, CVCL Accession (e.g., HEK293, CVCL_0045)
                    return best_match.get('identifier'), best_match.get('accession')
            elif res.status_code == 429:
                time.sleep(2)
            else:
                break
        except Exception as e:
            time.sleep(1)
            
    return None, None

def get_uniprot_by_gene(gene_symbol, taxonomy):
    """
    기존에 작성하신 UniProt 조회 코드 (최적화)
    """
    tax = str(taxonomy).strip()
    if pd.isna(tax) or tax.lower() == 'not specified' or tax == '':
        tax = 'Homo sapiens'
        
    query = f"(gene_exact:{gene_symbol}) AND (organism_name:\"{tax}\") AND (reviewed:true)"
    encoded_query = urllib.parse.quote(query)
    url = f"https://rest.uniprot.org/uniprotkb/search?query={encoded_query}&format=json&size=1"
    
    for attempt in range(3):
        try:
            res = requests.get(url, timeout=10)
            if res.status_code == 200:
                data = res.json()
                if 'results' in data and len(data['results']) > 0:
                    return data['results'][0]['primaryAccession']
                else:
                    if tax != 'Homo sapiens':
                        return get_uniprot_by_gene(gene_symbol, 'Homo sapiens')
                    return None
            elif res.status_code == 429:
                time.sleep(2)
            else:
                break
        except Exception as e:
            time.sleep(1)
            
    return None

In [ ]:
# ==========================================
# 2. Main Processing Logic
# ==========================================
def standardize_db(df):
    print(f"Total entries loaded: {len(df)}")
    
    # ---------------------------------------------------------
    # 1. Disease Normalization
    # ---------------------------------------------------------
    unique_diseases = df['Disease_Context'].dropna().unique()
    unique_diseases = [d for d in unique_diseases if d != 'N/A']
    
    print(f"\n[1] Fetching OLS Ontologies for {len(unique_diseases)} unique Diseases...")
    disease_map = {}
    disease_id_map = {}
    
    for d in tqdm(unique_diseases, desc='Mapping Diseases'):
        std_name, std_id = get_disease_ontology(d)
        disease_map[d] = std_name if std_name else d
        disease_id_map[d] = std_id if std_id else "N/A"
        
    df['Disease_Context_Std'] = df['Disease_Context'].map(disease_map).fillna("N/A")
    df['Disease_Ontology_ID'] = df['Disease_Context'].map(disease_id_map).fillna("N/A")

    # ---------------------------------------------------------
    # 2. Cell Line Normalization
    # ---------------------------------------------------------
    unique_cells = df['Cell_Line'].dropna().unique()
    unique_cells = [c for c in unique_cells if c != 'N/A']
    
    print(f"\n[2] Fetching Cellosaurus IDs for {len(unique_cells)} unique Cell Lines...")
    cell_map = {}
    cell_id_map = {}
    
    for c in tqdm(unique_cells, desc='Mapping Cell Lines'):
        std_name, std_id = get_cell_line_ontology(c)
        cell_map[c] = std_name if std_name else c
        cell_id_map[c] = std_id if std_id else "N/A"
        
    df['Cell_Line_Std'] = df['Cell_Line'].map(cell_map).fillna("N/A")
    df['Cell_Line_CVCL_ID'] = df['Cell_Line'].map(cell_id_map).fillna("N/A")

    # ---------------------------------------------------------
    # 3. UniProt AC Normalization
    # ---------------------------------------------------------
    # Target_Receptor가 존재하나 Accession이 없거나 지정되지 않은(Not specified) 경우 추출
    needs_mapping = df['Accession'].isna() | (df['Accession'] == 'Not specified')
    missing_uniprot_df = df[needs_mapping]
    
    unique_genes = missing_uniprot_df[['Target_Receptor', 'Taxonomy']].drop_duplicates()
    unique_genes = unique_genes[unique_genes['Target_Receptor'].notna() & (unique_genes['Target_Receptor'] != '')]
    
    print(f"\n[3] Fetching UniProt ACs for {len(unique_genes)} unique Gene/Taxonomy pairs...")
    uniprot_map = {}
    
    for _, row in tqdm(unique_genes.iterrows(), total=len(unique_genes), desc='Mapping UniProt'):
        gene = str(row['Target_Receptor']).strip()
        tax = str(row['Taxonomy']).strip()
        if len(gene) >= 2:
            ac = get_uniprot_by_gene(gene, tax)
            uniprot_map[(gene, tax)] = ac if ac else "Not found"
            
    # Apply UniProt mapping back to dataframe
    def update_ac(row):
        if pd.isna(row['Accession']) or row['Accession'] == 'Not specified':
            return uniprot_map.get((str(row['Target_Receptor']).strip(), str(row['Taxonomy']).strip()), row['Accession'])
        return row['Accession']
        
    df['Accession'] = df.apply(update_ac, axis=1)
    
    return df

In [ ]:
# ==========================================
# 3. Execution Block
# ==========================================
if __name__ == "__main__":
    input_csv = './Output/DB/PubChem/NAR/aid_information_llm_parsed_v2.csv'
    output_csv = './Output/DB/PubChem/NAR/aid_information_standardized.csv'
    
    if os.path.exists(input_csv):
        df_raw = pd.read_csv(input_csv)
        df_standardized = standardize_db(df_raw)
        
        # Save output
        df_standardized.to_csv(output_csv, index=False)
        print(f"\n✅ All normalizations complete. Saved to '{output_csv}'")
    else:
        print(f"Error: Could not find '{input_csv}'.")

In [ ]:
aid = pd.read_csv('./Output/DB/PubChem/NAR/aid_information_standardized.csv')

In [ ]:
aid